In [10]:
%%capture
from pyfiles.preamble import *
setup_notebook()

get_ipython().run_line_magic('run', '1_create_scenario.ipynb')

# Optimiser — finding endogenous capacities

This notebook minimises **total annual system costs** over a set of user-defined capacity variables using Nelder-Mead optimisation. It builds directly on the scenario parameters defined in `1_create_scenario.ipynb`.

**Workflow**
1. Pick the capacity variables to optimise (`x0`) and set starting values
2. Optionally constrain variables with lower/upper bounds
3. Set a maximum electricity import and a run budget (`max_evaluations`)
4. Run the solver for each scenario — results summarised at the bottom

Browse `pyfiles/input_variables.py` for available parameter names.

### 1. Choice variables

Each entry is `'parameter_name': initial_value`. The solver will optimise over all listed variables simultaneously. Browse `pyfiles/input_variables.py` for available names.

In [11]:
x0 = {
    'input_cap_pp2_el'          : 4_500.0,   # MW
    # 'input_cap_ELTtrans_el'     : 4_000.0,   # MW
    # 'input_H2storage_trans_cap' : 50.0,      # MW
}

### 2. Bounds *(optional)*

Constrain any choice variable with `(lower, upper)` — use `None` for no bound on that side. Variables not listed here are unconstrained.

In [12]:
bounds = {
    'input_cap_ELTtrans_el'     : (3200.0, None),
    'input_H2storage_trans_cap' : (32.0,   None),
}

### 3. Solver settings

- **`import_limit_twh`** — maximum annual electricity imports allowed (TWh); violations are penalised. Set to `0` to require self-sufficiency.
- **`max_evaluations`** — budget of EnergyPLAN calls. More complex problems (more variables, tighter bounds) need more evaluations to converge; start low to test, then increase.

In [13]:
import_limit_twh  = 0.5    # TWh — electricity import constraint
max_evaluations   = 40    

### 4. Run

In [14]:
# base
solver.configure(
    'base', base_params, base_case_params, shock_case_params,
    x0=x0, bounds=bounds, ref_file=ref_path,
    import_limit_twh=import_limit_twh,
)
out_base = solver.run(max_evaluations=max_evaluations)

# shock
solver.configure(
    'shock', base_params, base_case_params, shock_case_params,
    x0=x0, bounds=bounds, ref_file=ref_path,
    import_limit_twh=import_limit_twh,
)
out_shock = solver.run(max_evaluations=max_evaluations)

EnergyPLAN cost minimiser  (case='base', 40 max evals)
  Choice vars (1): cap_pp2_el=4500
  Import limit : 0.5 TWh
  Bound: input_cap_ELTtrans_el  >= 3200.0
  Bound: input_H2storage_trans_cap  >= 32.0
 *[  1] cap_pp2_el= 4500.0  → cost=23333.00 M EUR  imports=0.230 TWh  (18.9s)
  [  2] cap_pp2_el= 4725.0  → cost=23345.00 M EUR  imports=0.180 TWh  (24.0s)
 *[  3] cap_pp2_el= 4275.0  → cost=23322.00 M EUR  imports=0.290 TWh  (21.4s)
 *[  4] cap_pp2_el= 3825.0  → cost=23298.00 M EUR  imports=0.450 TWh  (28.8s)
  [  5] cap_pp2_el= 3150.0  → cost=23262.00 M EUR  imports=0.770 TWh  (27.5s)  [IMPORT 0.77 > 0.5 TWh!]
  [  6] cap_pp2_el= 3993.8  → cost=23307.00 M EUR  imports=0.380 TWh  (18.3s)
  [  7] cap_pp2_el= 3656.2  → cost=23289.00 M EUR  imports=0.520 TWh  (13.1s)  [IMPORT 0.52 > 0.5 TWh!]
  [  8] cap_pp2_el= 3867.2  → cost=23300.00 M EUR  imports=0.430 TWh  (12.7s)
 *[  9] cap_pp2_el= 3782.8  → cost=23296.00 M EUR  imports=0.460 TWh  (12.7s)
 *[ 10] cap_pp2_el= 3698.4  → cost=23291.00 M

### 5. Results

In [15]:
results = {'Base': out_base, 'Shock': out_shock}

rows = []
for name, out in results.items():
    row = {'Scenario': name, 'Total annual costs (M EUR)': round(out['best_cost'], 1)}
    for param, val in zip(x0.keys(), out['best_x']):
        from pyfiles.input_variables import labels as _inp_labels
        col = _inp_labels.get(param, param)
        row[col] = round(val, 1)
    rows.append(row)

pd.DataFrame(rows).set_index('Scenario')

,Total annual costs (M EUR),Peak load plant (PP2) capacity (MW)
Scenario,,
Base,23291.0,3698.4
Shock,23730.0,4148.4
